<a href="https://www.dna-evolutions.com/" target="_blank"><img src="https://www.dna-evolutions.com/images/dna_logo.png" width="200" title="DNA-Evolutions" alt="DNA-Evolutions"></a>

# JOpt TourOptimizer - Job-Based Optimization (Fire-and-Forget)

This notebook demonstrates how to submit an optimization as an **asynchronous job** that persists
its result to a database. Unlike the synchronous `start_run` flow, a job runs in the background
and stores its result automatically -- you can retrieve it later.

**Workflow:**
1. Build a `RestOptimization` input with persistence settings.
2. Submit via `create_job` (POST /api/v1/jobs) -- returns a `job_id` immediately.
3. Poll for status/progress, or wait and fetch the result later.
4. Retrieve the full result via `get_job_result`.

**Prerequisites:**
- A running TourOptimizer instance (>= 1.3.5) **with database support** on `http://localhost:8081`  
  See the [REST Server Docs](https://dna-evolutions.com/docs/learn-and-explore/rest/rest-server-touroptimizer) for database setup.
- This project installed: `pip install -e .` from the repository root.

**Documentation:** [Quickstart Guide](https://dna-evolutions.com/docs/getting-started/quickstart/jopt_sandboxes_quickstart) | [REST Server Docs](https://dna-evolutions.com/docs/learn-and-explore/rest/rest-server-touroptimizer)

## 1. Setup and Health Check

In [1]:
from util.tour_optimizer_rest_caller import TourOptimizerRestCaller
from util.tour_optimizer_endpoints import Endpoints

# Choose the correct URL for your setup:
# - Running locally (outside Docker):  Endpoints.LOCAL_SWAGGER_TOUROPTIMIZER_URL        -> http://localhost:8081
# - Running inside the Docker sandbox:  Endpoints.LOCAL_SWAGGER_TOUROPTIMIZER_FROM_DOCKER_URL -> http://host.docker.internal:8081
TOUR_OPTIMIZER_URL = Endpoints.LOCAL_SWAGGER_TOUROPTIMIZER_FROM_DOCKER_URL

caller = TourOptimizerRestCaller(tour_optimizer_url=TOUR_OPTIMIZER_URL)

status = caller.health()
print(f"Service status: {status}")

Checking Health
'http://host.docker.internal:8081'
Service status: description=None status='UP'


## 2. Build the Optimization Input

Same as the synchronous example -- create nodes, resources, and a `RestOptimization` object.

In [2]:
from util.test_position_input import TestPositionsInput
from util.test_rest_optimization_creator import TestRestOptimizationCreator

node_positions = TestPositionsInput.default_sydney_node_positions()
resource_positions = TestPositionsInput.default_sydney_resource_positions()

opti = TestRestOptimizationCreator.default_touroptimizer_position_test_input(
    node_positions,
    resource_positions,
    node_relations=[],
    element_connections=[],
    json_license=None
)

opti.ident = "JUPYTER_JOB_EXAMPLE"

print(f"Optimization input: {len(opti.nodes)} nodes, {len(opti.resources)} resources")

Optimization input: 5 nodes, 4 resources


## 3. Configure Persistence Settings

Job-based optimizations require persistence settings that control:
- **What to persist**: full config or result only, with or without connections.
- **Stream persistence**: whether to store progress, status, warnings, and errors.
- **Encryption**: optionally encrypt the result with a secret (metadata stays unencrypted).
- **Expiry**: auto-delete the stored result after a given duration.
- **Creator**: a label to tag and later search for your jobs.

In [3]:
from util.test_faf_helper import TestFAFHelper

# Encryption secret -- leave empty for unencrypted storage.
# WARNING: The secret is NOT stored by DNA Evolutions. If lost, the result CANNOT be restored.
secret = ""

# Creator label -- used to search for jobs later
creator = "JUPYTER_CREATOR"

# Build persistence settings
persistence_settings = TestFAFHelper.default_optimization_persistence_settings(secret=secret)

# Apply to the optimization
if opti.extension:
    opti.extension.persistence_setting = persistence_settings
    opti.extension.creator_setting = TestFAFHelper.default_creator_settings(creator)

print(f"Persistence configured:")
print(f"  Creator:    {creator}")
print(f"  Encrypted:  {'Yes' if secret else 'No'}")
print(f"  Expiry:     {persistence_settings.mongo_settings.expiry if persistence_settings.mongo_settings else 'N/A'}")

Persistence configured:
  Creator:    JUPYTER_CREATOR
  Encrypted:  No
  Expiry:     PT48H


## 4. Submit the Job

Call `create_job` to submit the optimization. The server returns a `JobAcceptedResponse`
with a `job_id` immediately -- the optimization runs asynchronously in the background.

The `tenant_id` scopes all data to a tenant (use any string for single-tenant / on-premise setups).

In [4]:
import touroptimizer_py_client
from touroptimizer_py_client.api.job_api import JobApi

tenant_id = "DEFAULT_TENANT"

# Initialize the Job API client
if caller.api_job_instance is None:
    caller._initialize_api_job_client()

# Submit the job
job_accepted = caller.api_job_instance.create_job(tenant_id, opti)

job_id = job_accepted.job_id
print(f"Job submitted!")
print(f"  job_id:       {job_id}")
print(f"  submitted_at: {job_accepted.submitted_at}")
print(f"  ident:        {job_accepted.ident}")
print(f"\nThe optimization is now running in the background.")
print(f"Use the job_id to poll for status or retrieve the result.")

'http://host.docker.internal:8081'
Job submitted!
  job_id:       e745e1b1-6710-4181-b637-ee7283b8d711
  submitted_at: 1775727244159
  ident:        JUPYTER_JOB_EXAMPLE

The optimization is now running in the background.
Use the job_id to poll for status or retrieve the result.


## 5. Poll for Progress (Optional)

While the job is running, you can check its progress and status.
This is optional -- you can also just wait and fetch the result directly in step 6.

In [5]:
import time

print(f"Polling progress for job {job_id}...\n")

for i in range(30):
    try:
        # Check status first
        status_list = caller.api_job_instance.get_job_status(job_id, tenant_id)
        if status_list and len(status_list) > 0:
            latest_status = status_list[0]
            desc = latest_status.desc if hasattr(latest_status, 'desc') else str(latest_status)
            code = latest_status.code if hasattr(latest_status, 'code') else None
            print(f"  [{i}] Status: {desc}")
            
            # Code 782 = "Optimization done" -- stop polling
            if code and code >= 780:
                print(f"\n  Job finished (code {code}).")
                break

        # Check progress
        progress = caller.api_job_instance.get_job_progress(job_id, tenant_id)
        if progress and len(progress) > 0:
            latest = progress[0]
            cur_progress = latest.cur_progress if hasattr(latest, 'cur_progress') else None
            caller_id = latest.caller_id if hasattr(latest, 'caller_id') else ""
            print(f"  [{i}] Progress: {cur_progress}% ({caller_id})")

    except touroptimizer_py_client.ApiException as e:
        print(f"  [{i}] Waiting... ({e.status})")
    
    time.sleep(3)
else:
    print("\nMax polling attempts reached. The job may still be running -- check the result in the next step.")

Polling progress for job e745e1b1-6710-4181-b637-ee7283b8d711...

  [0] Status: Optimization done

  Job finished (code 782).


## 6. Retrieve the Result

Fetch the completed optimization result from the database using `get_job_result`.
This call blocks until the result is available (up to `time_out`).

If the job was submitted with encryption, pass the same `secret` via `x_encryption_secret`.

In [6]:
# Fetch the result (blocks until available, up to time_out)
result = caller.api_job_instance.get_job_result(
    job_id,
    tenant_id,
    x_encryption_secret=secret if secret else None,
    time_out="PT2M"
)

if result and result.extension:
    text_solution = result.extension.text_solution
    if text_solution:
        print("=== Text Solution ===")
        print(text_solution.text_solution)
else:
    print("No result available yet. The job may still be running -- try again or increase time_out.")

=== Text Solution ===

-----------------------------------------------------------
---------------------- RUN --------------------------------
-----------------------------------------------------------
 Optimization-IDENT      : JUPYTER_JOB_EXAMPLE

-----------------------------------------------------------
--------------------- RESULTS -----------------------------
-----------------------------------------------------------
 Number of Route         : 20
 Number of Route (sched.): 3
 Total Route Elements    : 9
 Cost                    : 17.258318335477234
-----------------------------------------------------------
 Total time        [h]   : 2
 Total idle time   [h]   : 0
 Total prod. time  [h]   : 2
 Total tran. time  [h]   : 0
 Total utilization [%]   : 97
 Total distance    [km]  : 4
 Termi. time       [h]   : 0
 Termi. distance   [km]  : 1

-----------------------------------------------------------
--------------------- ROUTES ------------------------------
---------------------

## 7. Search for Jobs in the Database

You can search for previously submitted jobs using `list_jobs` with a `DatabaseInfoSearch` filter.
This is useful for finding jobs by creator, ident, or date range.

In [7]:
from touroptimizer_py_client.models.database_info_search import DatabaseInfoSearch

search = DatabaseInfoSearch()
search.creator = creator  # search by the creator we used above

jobs = caller.api_job_instance.list_jobs(search, x_tenant_id=tenant_id)

if jobs:
    print(f"Found {len(jobs)} job(s):\n")
    for job_info in jobs:
        print(f"  id:           {job_info.id}")
        print(f"  job_id:       {job_info.job_id}")
        print(f"  ident:        {job_info.ident}")
        print(f"  creator:      {job_info.creator}")
        print(f"  upload_date:  {job_info.upload_date}")
        print(f"  content_type: {job_info.content_type}")
        print(f"  encrypted:    {job_info.encrypted}")
        print(f"  expire_at:    {job_info.expire_at}")
        print()
else:
    print("No jobs found.")

Found 3 job(s):

  id:           69d7716353ad28e274da0938
  job_id:       3ffbdeac-0c9f-462d-88ac-1f2d40cfa9e8
  ident:        JUPYTER_JOB_EXAMPLE
  creator:      JUPYTER_CREATOR
  upload_date:  2026-04-09 09:29:07.264000+00:00
  content_type: application/x-bzip2
  encrypted:    False
  expire_at:    2026-04-11 09:29:07.261000+00:00

  id:           69d7721153ad28e274da093c
  job_id:       9917a495-a49f-492d-9860-18583e64fd93
  ident:        JUPYTER_JOB_EXAMPLE
  creator:      JUPYTER_CREATOR
  upload_date:  2026-04-09 09:32:01.991000+00:00
  content_type: application/x-bzip2
  encrypted:    False
  expire_at:    2026-04-11 09:32:01.988000+00:00

  id:           69d7728d53ad28e274da0940
  job_id:       e745e1b1-6710-4181-b637-ee7283b8d711
  ident:        JUPYTER_JOB_EXAMPLE
  creator:      JUPYTER_CREATOR
  upload_date:  2026-04-09 09:34:05.799000+00:00
  content_type: application/x-bzip2
  encrypted:    False
  expire_at:    2026-04-11 09:34:05.796000+00:00



## 8. Webhooks: Get Notified Instead of Polling

Instead of polling for progress or results, you can configure a **completion webhook**.
When enabled, the server sends a POST request to your webhook URL as soon as the job
completes or fails. The payload contains `jobId`, `tenantId`, `status`, and `completedAt` --
no optimization data, keeping the callback lightweight.

To enable webhooks, set them in the `MongoOptimizationPersistenceSetting`:

```python
mongo_settings.completion_webhook_url = "https://your-server.com/webhook/jopt"
mongo_settings.completion_webhook_secret = "your-hmac-secret"  # optional, for HMAC-SHA256 signature
```

When a `completion_webhook_secret` is provided, the server signs the payload with HMAC-SHA256
and includes an `X-JOpt-Signature` header (`sha256=<hex>`). Your receiver should verify this
signature before trusting the payload.

**URL validation** depends on the server's webhook-validation policy:
- **strict** (SaaS default): only public HTTPS URLs
- **relaxed** (on-premise): HTTP and private networks allowed
- **none** (local development): any URL accepted

Loopback addresses (127.x, ::1) are always rejected.

See the [REST Server Documentation](https://dna-evolutions.com/docs/learn-and-explore/rest/rest-server-touroptimizer) for more details.

In [8]:
# Example: How to configure a webhook (not executed -- adjust URL to your setup)

from util.test_faf_helper import TestFAFHelper

webhook_persistence = TestFAFHelper.default_optimization_persistence_settings(secret="")

if webhook_persistence.mongo_settings:
    # Set the webhook URL -- the server will POST here when the job completes
    webhook_persistence.mongo_settings.completion_webhook_url = "https://your-server.com/webhook/jopt"
    
    # Optional: HMAC-SHA256 secret for verifying the webhook signature
    webhook_persistence.mongo_settings.completion_webhook_secret = "your-hmac-secret"

print("Webhook settings configured (not submitted -- adjust the URL first):")
print(f"  URL:    {webhook_persistence.mongo_settings.completion_webhook_url}")
print(f"  Secret: {'***' if webhook_persistence.mongo_settings.completion_webhook_secret else 'None (unsigned)'}")
print(f"\nApply to opti.extension.persistence_setting before calling create_job().")

Webhook settings configured (not submitted -- adjust the URL first):
  URL:    https://your-server.com/webhook/jopt
  Secret: ***

Apply to opti.extension.persistence_setting before calling create_job().
